# 1. Reward Model

In this section, we train a **reward model** to evaluate the quality or relevance of generated content.  
We skip the earlier stages of the LLM pipeline — **(1) pretraining** and **(2) supervised fine-tuning** — and start from an already **instruction-tuned model**.

---

### Objective

The goal is to learn a reward function  
$$
r_\phi(x, y)
$$  
that assigns a scalar value to a model output $y$ given an input $x$.  

Here, $x \sim \mathcal{D}$ represents a sample drawn from the data distribution,  and $ y \sim \pi_\theta(y \mid x) $ is a response generated by the language model.
This value represents how *preferred* or *relevant* the output is, acting as a proxy for human feedback.


### Approach

A common method is to frame reward modeling as a **regression task**, where the model predicts an **unbounded scalar reward**. 
 
We use a pretrained language model $\pi_\theta(y \mid x)$ and attach a final linear layer with a single neuron, which will be trained to output the predicted reward value $r_\phi(x, y)$ from the positive and negative prompts in the prompt database $\mathcal{D}$.

---
### Training the Reward Model

The reward model $r_\phi(x, y)$ is trained to predict human (or synthetic) preferences over pairs of model outputs.

#### Preference-Based Objective

Given a prompt $x$ and two candidate responses $(y^+, y^-)$,  
where $y^+$ is preferred over $y^-$ according to human feedback,  
the model should assign a higher reward to $y^+$:

$$
r_\phi(x, y^+) > r_\phi(x, y^-)
$$

To enforce this, we use a **pairwise logistic loss** (used in RLHF, e.g. in InstructGPT):

$$
\mathcal{L}_{\text{RM}}(\phi)
= - \mathbb{E}_{(x, y^+, y^-) \sim \mathcal{D}}
  \left[
    \log \sigma\!\left(r_\phi(x, y^+) - r_\phi(x, y^-)\right)
  \right]
$$

where $\sigma(\cdot)$ is the sigmoid function:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

This encourages the model to assign a higher score to the preferred completion.

---

#### Intuition

- If $r_\phi(x, y^+) \gg r_\phi(x, y^-)$,  
  then $\sigma(r_\phi(x, y^+) - r_\phi(x, y^-)) \approx 1$,  
  and the loss is small.  
- If the model ranks them incorrectly, the loss is large.  

---

#### Alternative (Regression) Objective

If explicit preference pairs are unavailable,  
the reward model can also be trained via regression to approximate scalar feedback values:

$$
\mathcal{L}_{\text{reg}}(\phi)
= \mathbb{E}_{(x, y, R) \sim \mathcal{D}}
  \left[ (r_\phi(x, y) - R)^2 \right]
$$

---

We will use pairwise as the training objective rather than the regression training loss.



In [2]:
import transformers
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, PreTrainedTokenizerBase
from transformers import AutoTokenizer, TrainingArguments, default_data_collator

from trl import RewardConfig, RewardTrainer
from peft import LoraConfig 
import pandas as pd 
import torch
from datasets import load_dataset, DatasetDict
from huggingface_hub import login
import os
from typing import Dict

/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


To oversimply the life-cycle of RLHF, we will use TRL hugging face library to see how this can performed easily with a high level of abstraction thanks to this library

In [3]:
SEED = 42
SHUFFLE_SEED = 42
HF_DATASET_ID = "eZWALT/rlhf_reward_data_raw"  
HUB_REPO_ID = "eZWALT/rlhf_reward_splits_raw"  
PUSH_TO_HUB = False


SEED = 42
SHUFFLE_SEED = 42
ds = load_dataset(HF_DATASET_ID, split="train")   


# 2) shuffle then split to 80/10/10
# First shuffle the entire dataset (important to get a random split)
ds = ds.shuffle(seed=SHUFFLE_SEED)

# Split 80/20 (train / rest)
train_test = ds.train_test_split(test_size=0.20, seed=SEED)
train_ds = train_test["train"]          # ~80%
rest_ds = train_test["test"]            # ~20%

# Split the rest into half/half -> validation/test = 10% each
val_test = rest_ds.train_test_split(test_size=0.5, seed=SEED)
val_ds = val_test["train"]              # ~10%
test_ds = val_test["test"]              # ~10%

# Put into DatasetDict
dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(dataset_dict)
print("Train / Val / Test sizes:", len(dataset_dict["train"]), len(dataset_dict["validation"]), len(dataset_dict["test"]))

# 3a) Save locally for later use (optional)
dataset_dict.save_to_disk("../data/hf_rlhf_splits")

# 3b) Push the new split dataset to the Hub (optional)
if PUSH_TO_HUB:
    dataset_dict.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed split dataset to hub at:", HUB_REPO_ID)


DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
})
Train / Val / Test sizes: 1200 150 150


Saving the dataset (1/1 shards): 100%|██████████| 150/150 [00:00<00:00, 21176.94 examples/s]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.bfloat16)

# Configuration for the Reward Model training loop
reward_config = RewardConfig(
    bf16=False,
    disable_dropout=False,
    remove_unused_columns=False,
    logging_steps=10,
    num_
)
# important to include the score head when base model is not a sequence classification model
lora_config = LoraConfig(
    modules_to_save=["score"],
)

processing_class = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

reward_trainer = RewardTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=reward_config,
    #peft_config=lora_config,
    processing_class=processing_class,
)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
reward_trainer.train()

/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.043400
20,0.991000
30,0.627100
40,0.774000
50,0.570700
60,0.474200
70,0.388300
80,0.439600
90,0.331600
100,0.286400


TrainOutput(global_step=450, training_loss=0.24140177408854166, metrics={'train_runtime': 72880.7739, 'train_samples_per_second': 0.049, 'train_steps_per_second': 0.006, 'total_flos': 0.0, 'train_loss': 0.24140177408854166, 'epoch': 3.0})

In [ ]:
metrics = reward_trainer.evaluate()
reward_trainer.log_metrics("eval", metrics)
reward_trainer.save_metrics("eval", metrics)

In [12]:
## Push to HuggingFace
repo_name = "SmolLM2-135M-Pedantic-Reward-Model"
#reward_trainer.push_to_hub(repo_name)

model.push_to_hub(repo_name)

Processing Files (1 / 1): 100%|██████████|  269MB /  269MB,  121MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/eZWALT/SmolLM2-135M-Pedantic-Reward-Model/commit/584824586d72db0ffd2c7304aef76af950368ddd', commit_message='Upload LlamaForSequenceClassification', commit_description='', oid='584824586d72db0ffd2c7304aef76af950368ddd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eZWALT/SmolLM2-135M-Pedantic-Reward-Model', endpoint='https://huggingface.co', repo_type='model', repo_id='eZWALT/SmolLM2-135M-Pedantic-Reward-Model'), pr_revision=None, pr_num=None)

### Visualize the architechture of the Reward Model

This reward model uses a $30$-layer SmolLM2 transformer architecture with $576$-dimensional token embeddings from a $49152$ vocabulary. The model processes sequences through rotary positional encodings (RoPE) and multi-head attention with $3$ heads using $\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$, followed by MLP blocks that expand $576→1536$ dimensions with $SiLU$ activation before projecting back. Each layer uses $RMSNorm$ for normalization. The final linear head $\text{Reward} = W \cdot h_{\text{final}}$ converts the last hidden state into a single reward score for RLHF training (that takes an embedding $\mathcal{R}^{576}$ and outputs $2$ values)

In [ ]:
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps

In [20]:
tokenizer = processing_class
normal_text = "This is a great movie and I loved every minute of it."
pedantic_text = "This cinematic production constitutes an exemplary fulfillment of its artistic aims, and I found the entire temporal duration of its presentation to be a source of unmitigated positive engagement."

inputs = tokenizer([normal_text, pedantic_text], return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
outputs


/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


SequenceClassifierOutputWithPast(loss=None, logits=tensor([[-0.9609, -3.0156],
        [ 1.1641, -0.7422]], dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)

# 2. Reinforcement Learning from Human Feedback (RLHF) 
After training our reward model, we are going to proceed into this preference alingment procedure by using this reward model to guide fine-tuning. Our end goal is to get the most pedantic version of the LLM we can possibly get. Note that here the bottleneck of this process will always be the reward model itself as it will be the "Critic" guiding training of the final model in the algorithms that make use of this reward model.


Now lets define briefly and dissect the hierarchy of reinforcement learning algorithms that we can use to optimize our model with the end goal of preference alignment.
**RLHF is essentially reward modelling (if needed) & model optimization (fine-tuning)** These optimization algorithms can be all categorized in 3 buckets:

1.
2.
3.

## 2.1 Proximal Policy Optimization (PPO)

## 2.2 Direct Policy Optimization (DPO)

## 2.3 Group Relative Policy Optimization (GRPO)